# SCRIPT 3 — MORPHOLOGICAL INTEGRITY ASSESSMENT
## Brahmaputra Morphodynamics Study (AP-GARSS 2026)

Extracts morphometric variables from 20 cross-sections for three sources:
1. Index-based weak label validation mask
2. Attention U-Net prediction (braidplain clipped)
3. SegFormer-B2 prediction (braidplain clipped)

Output: Excel file with 5 sheets + R² regression figures for model selection.

**Workflow:**
1. Run all cells to extract prediction metrics and save Excel
2. Complete manual measurements in Excel → Manual_Measurements sheet
3. Run R² regression cell to compare all three sources against manual
4. Select model with highest R² for Script 4 multi-temporal inference

## 0. Install and Imports

In [ ]:
!pip install geopandas rasterio openpyxl shapely -q

import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from shapely.geometry import LineString
from openpyxl import Workbook, load_workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side)
from openpyxl.utils import get_column_letter
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr, linregress

from google.colab import drive
drive.mount('/content/drive')
print('Imports complete.')

## 1. Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────
ROOT           = '/content/drive/MyDrive/BRAHMA_MORPHO'
CROSS_SEC_PATH = f'{ROOT}/CS/CS.shp'  # update if .shp
VAL_MASK_PATH  = f'{ROOT}/inference/scenes/inference_2025_validation_mask.tif'
ATTN_PRED_PATH = f'{ROOT}/models/attention_unet/attention_unet_pred_2025_braidplain.tif'
SEG_PRED_PATH  = f'{ROOT}/models/segformer/segformer_pred_2025_braidplain.tif'
OUTPUT_EXCEL   = f'{ROOT}/validation/morphological_integrity_assessment1.xlsx'
FIGURES_DIR    = f'{ROOT}/figures/validation'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_EXCEL), exist_ok=True)

RESOLUTION     = 10       # metres per pixel
N_PROFILE_PTS  = 1000     # sampling points along cross-section
CS_ID_FIELD    = 'CS_id'  # attribute field in your shapefile

# ── Manual measurements from spreadsheet ─────────────────────────────────
# Pre-populated from your photo — fill remaining None values as you measure
MANUAL_DATA = {
    'CS_01': {'braidplain_width_km': 9.9381,  'channel_width_km': 2.251,
              'bar_width_km': 7.587,   'veg_bar_width_km': 2.668,
              'n_channel_threads': 7,  'n_bar_units': 8},
    'CS_02': {'braidplain_width_km': 13.366,  'channel_width_km': 1.6,
              'bar_width_km': 11.724,  'veg_bar_width_km': 7.149,
              'n_channel_threads': 4,  'n_bar_units': 4},
    'CS_03': {'braidplain_width_km': 10.435,  'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_04': {'braidplain_width_km': 8.2913,  'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_05': {'braidplain_width_km': 11.3153, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_06': {'braidplain_width_km': 10.3035, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_07': {'braidplain_width_km': 11.0107, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_08': {'braidplain_width_km': 8.46306, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_09': {'braidplain_width_km': 6.31894, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_10': {'braidplain_width_km': 9.77891, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_11': {'braidplain_width_km': 9.61232, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_12': {'braidplain_width_km': 10.2226, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_13': {'braidplain_width_km': 10.3316, 'channel_width_km': None,
              'bar_width_km': None,    'veg_bar_width_km': None,
              'n_channel_threads': None, 'n_bar_units': None},
    'CS_14': {'braidplain_width_km': 13.842,  'channel_width_km': 2.454,
              'bar_width_km': 11.378,  'veg_bar_width_km': 4.864,
              'n_channel_threads': 12, 'n_bar_units': 13},
    'CS_15': {'braidplain_width_km': 7.3515,  'channel_width_km': 2.207,
              'bar_width_km': 5.163,   'veg_bar_width_km': 1.558,
              'n_channel_threads': 6,  'n_bar_units': 5},
    'CS_16': {'braidplain_width_km': 3.13824, 'channel_width_km': 1.434,
              'bar_width_km': 1.697,   'veg_bar_width_km': 1.259,
              'n_channel_threads': 3,  'n_bar_units': 3},
    'CS_17': {'braidplain_width_km': 1.539,   'channel_width_km': 0.61,
              'bar_width_km': 0.897,   'veg_bar_width_km': 0.03,
              'n_channel_threads': 2,  'n_bar_units': 4},
}

COLS = ['CS_id', 'braidplain_width_km', 'channel_width_km',
        'bar_width_km', 'veg_bar_width_km', 'bare_bar_width_km',
        'n_channel_threads', 'n_bar_units',
        'bar_channel_ratio', 'active_fraction']

MANUAL_COLS = ['CS_id', 'braidplain_width_km', 'channel_width_km',
               'bar_width_km', 'veg_bar_width_km',
               'n_channel_threads', 'n_bar_units']

COL_HEADERS = {
    'CS_id'               : 'Cross-Section ID',
    'braidplain_width_km' : 'Braidplain Width (km)',
    'channel_width_km'    : 'Channel Width (km)',
    'bar_width_km'        : 'Total Bar Width (km)',
    'veg_bar_width_km'    : 'Vegetated Bar Width (km)',
    'bare_bar_width_km'   : 'Bare Bar Width (km)',
    'n_channel_threads'   : 'N Channel Threads',
    'n_bar_units'         : 'N Bar Units',
    'bar_channel_ratio'   : 'Bar/Channel Ratio',
    'active_fraction'     : 'Active Fraction',
}

print('Configuration set.')

## 2. Extraction Functions

In [ ]:
# def sample_raster_along_line(arr, transform, geom, n_points=N_PROFILE_PTS):
#     distances = np.linspace(0, geom.length, n_points)
#     values    = []
#     for d in distances:
#         pt       = geom.interpolate(d)
#         col, row = ~transform * (pt.x, pt.y)
#         col, row = int(col), int(row)
#         if 0 <= row < arr.shape[0] and 0 <= col < arr.shape[1]:
#             values.append(int(arr[row, col]))
#         else:
#             values.append(0)
#     return np.array(values)


# def count_segments(profile, class_val):
#     in_seg = False
#     count  = 0
#     for v in profile:
#         if v == class_val and not in_seg:
#             in_seg = True
#             count += 1
#         elif v != class_val:
#             in_seg = False
#     return count


# def profile_to_metrics(profile, cs_id, res=RESOLUTION):
#     active_idx = np.where(profile > 0)[0]
#     bp_width   = float((active_idx[-1] - active_idx[0]) * res) / 1000 \
#                  if len(active_idx) >= 2 else 0.0
#     ch_px  = int(np.sum(profile == 1))
#     bar_px = int(np.sum(profile == 2))
#     veg_px = int(np.sum(profile == 3))
#     return {
#         'CS_id'               : cs_id,
#         'braidplain_width_km' : round(bp_width, 4),
#         'channel_width_km'    : round(ch_px  * res / 1000, 4),
#         'bar_width_km'        : round((bar_px + veg_px) * res / 1000, 4),
#         'veg_bar_width_km'    : round(veg_px * res / 1000, 4),
#         'bare_bar_width_km'   : round(bar_px * res / 1000, 4),
#         'n_channel_threads'   : count_segments(profile, 1),
#         'n_bar_units'         : count_segments(profile, 2) + count_segments(profile, 3),
#         'bar_channel_ratio'   : round((bar_px + veg_px) / max(ch_px, 1), 3),
#         'active_fraction'     : round(float(np.sum(profile > 0)) / max(len(profile), 1), 4),
#     }


# def extract_all_crosssections(raster_path, cs_gdf, label):
#     results = []
#     with rasterio.open(raster_path) as src:
#         arr       = src.read(1)
#         transform = src.transform
#         cs_repr   = cs_gdf.to_crs(src.crs) if cs_gdf.crs != src.crs else cs_gdf
#     for _, row in tqdm(cs_repr.iterrows(), total=len(cs_repr), desc=f'  {label}'):
#         cs_id   = str(row[CS_ID_FIELD])
#         profile = sample_raster_along_line(arr, transform, row.geometry)
#         results.append(profile_to_metrics(profile, cs_id))
#     return pd.DataFrame(results)


# print('Extraction functions defined.')

## 3. Excel Styling

In [ ]:
THIN  = Border(left=Side(style='thin'),  right=Side(style='thin'),
               top=Side(style='thin'),   bottom=Side(style='thin'))
FILLS = {
    'header' : PatternFill('solid', start_color='1F3864'),
    'subhdr' : PatternFill('solid', start_color='2E75B6'),
    'alt'    : PatternFill('solid', start_color='D6E4F0'),
    'manual' : PatternFill('solid', start_color='E2EFDA'),
    'weak'   : PatternFill('solid', start_color='FFF2CC'),
    'attn'   : PatternFill('solid', start_color='FCE4D6'),
    'seg'    : PatternFill('solid', start_color='EAD1DC'),
}


def style_sheet(ws, df, title, note=''):
    ncols = len(df.columns)
    ws.merge_cells(f'A1:{get_column_letter(ncols)}1')
    ws['A1'].value     = title
    ws['A1'].font      = Font(bold=True, size=13, color='FFFFFF', name='Arial')
    ws['A1'].fill      = FILLS['header']
    ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 28

    hrow = 2
    if note:
        ws.merge_cells(f'A2:{get_column_letter(ncols)}2')
        ws['A2'].value     = note
        ws['A2'].font      = Font(italic=True, size=9, color='595959', name='Arial')
        ws['A2'].fill      = PatternFill('solid', start_color='F2F2F2')
        ws['A2'].alignment = Alignment(horizontal='left', vertical='center')
        ws.row_dimensions[2].height = 18
        hrow = 3

    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(row=hrow, column=ci)
        c.value     = COL_HEADERS.get(col, col)
        c.font      = Font(bold=True, color='FFFFFF', size=10, name='Arial')
        c.fill      = FILLS['subhdr']
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border    = THIN
    ws.row_dimensions[hrow].height = 36

    for ri, (_, row) in enumerate(df.iterrows()):
        er   = hrow + 1 + ri
        fill = FILLS['alt'] if ri % 2 == 0 \
               else PatternFill('solid', start_color='FFFFFF')
        for ci, col in enumerate(df.columns, 1):
            c        = ws.cell(row=er, column=ci)
            val      = row[col]
            c.value  = None if (isinstance(val, float) and np.isnan(val)) else val
            c.font   = Font(size=10, name='Arial')
            c.fill   = fill
            c.border = THIN
            c.alignment = Alignment(horizontal='center' if col != 'CS_id'
                                    else 'left', vertical='center')
            if col in ['n_channel_threads','n_bar_units']:
                c.number_format = '0'
            elif col == 'active_fraction':
                c.number_format = '0.00%'
            elif col != 'CS_id':
                c.number_format = '0.0000'
        ws.row_dimensions[er].height = 18

    widths = {'CS_id':14,'braidplain_width_km':22,'channel_width_km':20,
              'bar_width_km':18,'veg_bar_width_km':20,'bare_bar_width_km':18,
              'n_channel_threads':18,'n_bar_units':14,
              'bar_channel_ratio':18,'active_fraction':16}
    for ci, col in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(ci)].width = widths.get(col, 18)
    ws.freeze_panes = ws.cell(row=hrow+1, column=2)


print('Styling functions defined.')

In [ ]:
def count_segments_filtered(profile, class_val, min_pixels=3):
    """
    Count segments of class_val with minimum length filter.
    Segments shorter than min_pixels are ignored as noise.
    min_pixels=3 corresponds to 30m minimum width at 10m resolution.
    """
    segments = []
    in_seg   = False
    seg_len  = 0

    for v in profile:
        if v == class_val:
            in_seg  = True
            seg_len += 1
        else:
            if in_seg and seg_len >= min_pixels:
                segments.append(seg_len)
            in_seg  = False
            seg_len = 0

    if in_seg and seg_len >= min_pixels:
        segments.append(seg_len)

    return len(segments)


def sum_segments_filtered(profile, class_val, min_pixels=3):
    """
    Sum pixel widths of segments passing minimum length filter.
    Excludes narrow noise segments from width calculation.
    """
    total_px = 0
    in_seg   = False
    seg_len  = 0

    for v in profile:
        if v == class_val:
            in_seg  = True
            seg_len += 1
        else:
            if in_seg and seg_len >= min_pixels:
                total_px += seg_len
            in_seg  = False
            seg_len = 0

    if in_seg and seg_len >= min_pixels:
        total_px += seg_len

    return total_px


def profile_to_metrics(profile, cs_id, line_length_m,
                        res_per_pt=None, min_seg_px=5):
    """
    Compute morphometric variables from 1D class profile.

    profile      : numpy array of class values along cross-section
    cs_id        : cross-section ID string
    line_length_m: actual line length in metres from geometry
    res_per_pt   : metres per sample point (line_length / n_points)
    min_seg_px   : minimum segment length in sample points to count
                   as real feature (default 3 = 30m at 10m resolution)
    """
    if res_per_pt is None:
        res_per_pt = line_length_m / len(profile)

    # ── Width calculations using filtered segments ─────────────────────
    # Only count segments >= min_seg_px sample points wide
    # This removes single-pixel noise and very narrow features
    # that are below the meaningful geomorphological scale

    ch_px_filt  = sum_segments_filtered(profile, 1, min_seg_px)
    bar_px_filt = sum_segments_filtered(profile, 2, min_seg_px)
    veg_px_filt = sum_segments_filtered(profile, 3, min_seg_px)

    ch_width  = round(ch_px_filt  * res_per_pt / 1000, 4)   # km
    bar_width = round((bar_px_filt + veg_px_filt) * res_per_pt / 1000, 4)
    veg_width = round(veg_px_filt * res_per_pt / 1000, 4)
    bare_width= round(bar_px_filt * res_per_pt / 1000, 4)

    # ── Thread and bar unit counts using filtered segments ─────────────
    n_threads  = count_segments_filtered(profile, 1, min_seg_px)
    n_bar_bare = count_segments_filtered(profile, 2, min_seg_px)
    n_bar_veg  = count_segments_filtered(profile, 3, min_seg_px)
    n_bar_units= n_bar_bare + n_bar_veg

    # ── Bar channel ratio ──────────────────────────────────────────────
    bar_ch_ratio = round((bar_px_filt + veg_px_filt) /
                          max(ch_px_filt, 1), 3)

    # ── Active fraction ────────────────────────────────────────────────
    # Fraction of total cross-section length covered by active classes
    total_active = ch_px_filt + bar_px_filt + veg_px_filt
    active_frac  = round(total_active / max(len(profile), 1), 4)

    return {
        'CS_id'               : cs_id,
        'channel_width_km'    : ch_width,
        'bar_width_km'        : bar_width,
        'veg_bar_width_km'    : veg_width,
        'bare_bar_width_km'   : bare_width,
        'n_channel_threads'   : n_threads,
        'n_bar_units'         : n_bar_units,
        'bar_channel_ratio'   : bar_ch_ratio,
        'active_fraction'     : active_frac,
    }


def extract_all_crosssections(raster_path, cs_gdf, label,
                               n_points=1000, min_seg_px=3):
    results = []
    with rasterio.open(raster_path) as src:
        arr       = src.read(1)
        transform = src.transform
        cs_repr   = cs_gdf.to_crs(src.crs) \
                    if cs_gdf.crs != src.crs else cs_gdf

    for _, row in tqdm(cs_repr.iterrows(),
                       total=len(cs_repr), desc=f'  {label}'):
        cs_id       = str(row[CS_ID_FIELD])
        geom        = row.geometry
        line_len_m  = geom.length
        res_per_pt  = line_len_m / n_points

        distances = np.linspace(0, line_len_m, n_points)
        profile   = []
        for d in distances:
            pt       = geom.interpolate(d)
            col, r   = ~transform * (pt.x, pt.y)
            col, r   = int(col), int(r)
            if 0 <= r < arr.shape[0] and 0 <= col < arr.shape[1]:
                profile.append(int(arr[r, col]))
            else:
                profile.append(0)

        profile = np.array(profile)
        metrics = profile_to_metrics(
            profile, cs_id, line_len_m,
            res_per_pt=res_per_pt,
            min_seg_px=min_seg_px
        )
        results.append(metrics)

    return pd.DataFrame(results)

In [ ]:
# Quick verification on CS_01 before running all cross-sections
cs_gdf  = gpd.read_file(CROSS_SEC_PATH)
cs_repr = cs_gdf.to_crs('EPSG:32646')
cs_01r  = cs_repr[cs_repr['CS_id'] == 'CS_01'].iloc[0]

with rasterio.open(ATTN_PRED_PATH) as src:
    arr       = src.read(1)
    transform = src.transform

n_points   = 1000
line_len_m = cs_01r.geometry.length
res_per_pt = line_len_m / n_points

distances = np.linspace(0, line_len_m, n_points)
profile   = []
for d in distances:
    pt     = cs_01r.geometry.interpolate(d)
    c, r   = ~transform * (pt.x, pt.y)
    c, r   = int(c), int(r)
    profile.append(int(arr[r, c]) if 0<=r<arr.shape[0]
                   and 0<=c<arr.shape[1] else 0)

profile = np.array(profile)

# Test different min_seg_px values to find best match
print('Manual CS_01: channel=2.251km  bar=7.587km  '
      'veg_bar=2.668km  threads=7  bar_units=8')
print()
for min_px in [2, 3, 4, 5]:
    m = profile_to_metrics(profile, 'CS_01', line_len_m,
                           res_per_pt=res_per_pt,
                           min_seg_px=min_px)
    print(f'min_seg_px={min_px}: '
          f'ch={m["channel_width_km"]:.3f}  '
          f'bar={m["bar_width_km"]:.3f}  '
          f'veg={m["veg_bar_width_km"]:.3f}  '
          f'threads={m["n_channel_threads"]}  '
          f'bar_units={m["n_bar_units"]}')

## 4. Run Extraction and Build Excel

In [ ]:
print('='*65)
print('MORPHOLOGICAL INTEGRITY ASSESSMENT')
print('='*65)

# ── Load cross-sections ───────────────────────────────────────────────────
print('\nLoading cross-sections...')
cs_gdf = gpd.read_file(CROSS_SEC_PATH)
cs_ids = sorted(cs_gdf[CS_ID_FIELD].astype(str).tolist())
print(f'  {len(cs_ids)} cross-sections: {cs_ids}')

# ── Extract from all three rasters ───────────────────────────────────────
print('\nExtracting metrics from rasters...')
df_weak = extract_all_crosssections(VAL_MASK_PATH,  cs_gdf, 'Weak Label')
df_attn = extract_all_crosssections(ATTN_PRED_PATH, cs_gdf, 'Attention U-Net')
df_seg  = extract_all_crosssections(SEG_PRED_PATH,  cs_gdf, 'SegFormer')

# ── Build manual dataframe ───────────────────────────────────────────────
manual_rows = []
for cs_id in cs_ids:
    row = {'CS_id': cs_id}
    row.update(MANUAL_DATA.get(cs_id,
               {k: None for k in MANUAL_COLS[1:]}))
    manual_rows.append(row)
df_manual = pd.DataFrame(manual_rows, columns=MANUAL_COLS)

# Define RASTER_COLS to exclude 'braidplain_width_km' as it's not generated by raster extraction
RASTER_COLS = [
    'CS_id', 'channel_width_km', 'bar_width_km', 'veg_bar_width_km',
    'bare_bar_width_km', 'n_channel_threads', 'n_bar_units',
    'bar_channel_ratio', 'active_fraction'
]

# ── Build Excel ──────────────────────────────────────────────────────────
print('\nBuilding Excel workbook...')
wb = Workbook()

ws1 = wb.active
ws1.title = 'Manual_Measurements'
style_sheet(ws1, df_manual,
            'Manual Measurements — NIR False Colour Visual Interpretation',
            note='Visual interpretation of 2025 Sentinel-2 NIR false colour. '
                 'Measurements in km. Empty = measurements in progress.')

ws2 = wb.create_sheet('Weak_Label')
style_sheet(ws2, df_weak[RASTER_COLS],
            'Index-Based Weak Label — Validation Mask (2025)',
            note='BSI/MSAVI2/NDVI index-based labels. '
                 'Source: inference_2025_validation_mask.tif')

ws3 = wb.create_sheet('Attention_UNet')
style_sheet(ws3, df_attn[RASTER_COLS],
            'Attention U-Net Prediction — Braidplain Clipped (2025)',
            note='Attention U-Net best checkpoint. '
                 'Source: attention_unet_pred_2025_braidplain.tif')

ws4 = wb.create_sheet('SegFormer')
style_sheet(ws4, df_seg[RASTER_COLS],
            'SegFormer-B2 Prediction — Braidplain Clipped (2025)',
            note='SegFormer-B2 best checkpoint. '
                 'Source: segformer_pred_2025_braidplain.tif')

# ── Summary comparison sheet ──────────────────────────────────────────────
ws5 = wb.create_sheet('Summary_Comparison')
ws5.merge_cells('A1:W1')
ws5['A1'].value     = 'Morphological Integrity — Cross-Section Comparison (All Sources)'
ws5['A1'].font      = Font(bold=True, size=13, color='FFFFFF', name='Arial')
ws5['A1'].fill      = FILLS['header']
ws5['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws5.row_dimensions[1].height = 28

vars_cmp = [
    ('channel_width_km',   'Channel Width (km)'),
    ('bar_width_km',       'Total Bar Width (km)'),
    ('veg_bar_width_km',   'Veg Bar Width (km)'),
    ('n_channel_threads',  'N Channel Threads'),
    ('n_bar_units',        'N Bar Units'),
]
sources_cmp = [
    ('Manual',     df_manual, 'E2EFDA'),
    ('Weak Label', df_weak,   'FFF2CC'),
    ('Attn U-Net', df_attn,   'FCE4D6'),
    ('SegFormer',  df_seg,    'EAD1DC'),
]

# Header rows
col = 2
ws5.cell(row=2, column=1, value='CS_id').font = Font(bold=True, name='Arial')
ws5.cell(row=2, column=1).fill = FILLS['subhdr']
ws5.cell(row=3, column=1, value='').fill = FILLS['subhdr']
for vk, vl in vars_cmp:
    start = col
    ws5.merge_cells(start_row=2, start_column=start,
                    end_row=2, end_column=start+len(sources_cmp)-1)
    c = ws5.cell(row=2, column=start)
    c.value     = vl
    c.font      = Font(bold=True, size=9, color='FFFFFF', name='Arial')
    c.fill      = FILLS['subhdr']
    c.alignment = Alignment(horizontal='center', wrap_text=True)
    for sn, _, sc in sources_cmp:
        c3       = ws5.cell(row=3, column=col)
        c3.value = sn
        c3.font  = Font(bold=True, size=8, name='Arial')
        c3.fill  = PatternFill('solid', start_color=sc)
        c3.alignment = Alignment(horizontal='center')
        c3.border    = THIN
        col += 1
ws5.row_dimensions[2].height = 28
ws5.row_dimensions[3].height = 20

# Data rows
for ri, cs_id in enumerate(cs_ids):
    er   = 4 + ri
    fill = FILLS['alt'] if ri % 2 == 0 \
           else PatternFill('solid', start_color='FFFFFF')
    c = ws5.cell(row=er, column=1, value=cs_id)
    c.font = Font(size=9, bold=True, name='Arial')
    c.fill = fill
    col = 2
    for vk, _ in vars_cmp:
        for _, df_src, sc in sources_cmp:
            c    = ws5.cell(row=er, column=col)
            match = df_src[df_src['CS_id'] == cs_id]
            if not match.empty and vk in match.columns:
                val = match.iloc[0][vk]
                c.value = None if (isinstance(val, float) and
                                   np.isnan(val)) else val
            c.font   = Font(size=9, name='Arial')
            c.fill   = PatternFill('solid', start_color=sc) \
                       if ri % 2 == 0 else fill
            c.border = THIN
            c.alignment = Alignment(horizontal='center')
            c.number_format = '0' if vk in ['n_channel_threads','n_bar_units'] \
                              else '0.0000'
            col += 1
    ws5.row_dimensions[er].height = 16

ws5.column_dimensions['A'].width = 14
for c in range(2, col):
    ws5.column_dimensions[get_column_letter(c)].width = 13
ws5.freeze_panes = ws5.cell(row=4, column=2)

wb.save(OUTPUT_EXCEL)
print(f'\n✓ Excel saved → {OUTPUT_EXCEL}')
print('\nSheets created:')
print('  1. Manual_Measurements  — fill remaining blank cells in Excel')
print('  2. Weak_Label           — index-based validation mask metrics')
print('  3. Attention_UNet       — Attention U-Net prediction metrics')
print('  4. SegFormer            — SegFormer-B2 prediction metrics')
print('  5. Summary_Comparison   — side-by-side all sources')
print('\nOnce manual measurements are complete run the R² cell below.')

## 5. R² Regression Analysis
**Run this cell only after completing all manual measurements in the Excel sheet.**

Compares Weak Label, Attention U-Net, and SegFormer against manual measurements.
Higher R² = better morphological integrity = preferred model for Script 4.

In [ ]:
# ── Load updated manual measurements from Excel ───────────────────────────
# After you fill in the Excel manually, reload it here
print('Loading manual measurements from Excel...')
df_manual_updated = pd.read_excel(OUTPUT_EXCEL,
                                   sheet_name='Manual_Measurements',
                                   header=2)  # skip title and note rows
# Clean column names
df_manual_updated.columns = MANUAL_COLS
df_manual_updated = df_manual_updated.dropna(subset=['CS_id'])
df_manual_updated['CS_id'] = df_manual_updated['CS_id'].astype(str)
print(f'  {len(df_manual_updated)} cross-sections loaded')
complete = df_manual_updated.dropna(subset=['channel_width_km'])
print(f'  {len(complete)} with complete measurements for regression')

# ── Variables to compare ──────────────────────────────────────────────────
VARS_REG = [
    ('braidplain_width_km', 'Braidplain\nWidth (km)'),
    ('channel_width_km',    'Channel\nWidth (km)'),
    ('bar_width_km',        'Total Bar\nWidth (km)'),
    ('veg_bar_width_km',    'Veg Bar\nWidth (km)'),
    ('n_channel_threads',   'N Channel\nThreads'),
]

SOURCES_REG = [
    ('Weak Label',   df_weak,  '#F4B942', 'o'),
    ('Attn U-Net',   df_attn,  '#E84393', 's'),
    ('SegFormer',    df_seg,   '#4393E8', '^'),
]

# ── R² computation ────────────────────────────────────────────────────────
def compute_r2(manual_df, pred_df, var):
    """Compute R² between manual and predicted values for a variable."""
    merged = manual_df[['CS_id', var]].merge(
        pred_df[['CS_id', var]], on='CS_id', suffixes=('_manual','_pred')
    ).dropna()
    if len(merged) < 3:
        return None, None, None, merged
    x = merged[f'{var}_manual'].values.astype(float)
    y = merged[f'{var}_pred'].values.astype(float)
    r, p    = pearsonr(x, y)
    r2      = r ** 2
    rmse    = np.sqrt(np.mean((y - x) ** 2))
    return r2, p, rmse, merged


# ── Plot R² regression panels ─────────────────────────────────────────────
fig = plt.figure(figsize=(22, 5 * len(VARS_REG)))
gs  = gridspec.GridSpec(len(VARS_REG), len(SOURCES_REG) + 1,
                         width_ratios=[1]*len(SOURCES_REG) + [0.7],
                         hspace=0.45, wspace=0.35)

fig.suptitle(
    'Morphological Integrity Assessment — R² Regression\n'
    'Manual Measurements vs Automated Extraction (20 Cross-Sections, 2025)',
    fontsize=13, fontweight='bold', y=1.01
)

summary_rows = []

for vi, (var, var_label) in enumerate(VARS_REG):
    r2_vals = {}

    for si, (src_name, df_src, color, marker) in enumerate(SOURCES_REG):
        ax = fig.add_subplot(gs[vi, si])

        r2, p, rmse, merged = compute_r2(complete, df_src, var)

        if r2 is None:
            ax.text(0.5, 0.5, 'Insufficient\ndata',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{src_name}\n{var_label}', fontsize=9)
            continue

        r2_vals[src_name] = r2
        x = merged[f'{var}_manual'].values.astype(float)
        y = merged[f'{var}_pred'].values.astype(float)

        # Scatter
        ax.scatter(x, y, color=color, marker=marker,
                   s=70, edgecolors='white', linewidth=0.8,
                   zorder=3, label='Cross-sections')

        # Labels for each point
        for _, row in merged.iterrows():
            ax.annotate(row['CS_id'],
                        (row[f'{var}_manual'], row[f'{var}_pred']),
                        fontsize=6, alpha=0.6,
                        xytext=(3, 3), textcoords='offset points')

        # 1:1 line
        lims = [min(x.min(), y.min()) * 0.9,
                max(x.max(), y.max()) * 1.1]
        ax.plot(lims, lims, 'k--', alpha=0.5, linewidth=1.2,
                label='1:1 line', zorder=2)

        # Regression line
        slope, inter, _, _, _ = linregress(x, y)
        x_line = np.linspace(lims[0], lims[1], 100)
        ax.plot(x_line, slope * x_line + inter, color=color,
                linewidth=1.8, alpha=0.8,
                label=f'Fit (R²={r2:.3f})', zorder=2)

        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel(f'Manual {var_label}', fontsize=8)
        ax.set_ylabel(f'Predicted {var_label}', fontsize=8)
        ax.set_title(
            f'{src_name} | {var_label}\n'
            f'R²={r2:.3f}  RMSE={rmse:.3f}  n={len(merged)}',
            fontsize=9, fontweight='bold'
        )
        ax.legend(fontsize=7, loc='upper left')
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')
        ax.tick_params(labelsize=8)

        summary_rows.append({
            'Variable': var_label.replace('\n',' '),
            'Source'  : src_name,
            'R²'      : round(r2, 4),
            'RMSE'    : round(rmse, 4),
            'N'       : len(merged),
            'p-value' : round(p, 4)
        })

    # R² bar chart — rightmost column
    ax_bar = fig.add_subplot(gs[vi, -1])
    if r2_vals:
        bars = ax_bar.barh(
            list(r2_vals.keys()),
            list(r2_vals.values()),
            color=[c for _, _, c, _ in SOURCES_REG if _ in r2_vals or True][:len(r2_vals)],
            edgecolor='white', height=0.5
        )
        ax_bar.set_xlim(0, 1)
        ax_bar.axvline(x=0.7, color='gray', linestyle='--',
                       alpha=0.6, linewidth=1)
        ax_bar.set_xlabel('R²', fontsize=9)
        ax_bar.set_title(f'{var_label}\nR² Summary', fontsize=9)
        ax_bar.grid(True, axis='x', alpha=0.3)
        for bar, r2v in zip(bars, r2_vals.values()):
            ax_bar.text(r2v + 0.02, bar.get_y() + bar.get_height()/2,
                        f'{r2v:.3f}', va='center', fontsize=9,
                        fontweight='bold')
        ax_bar.tick_params(labelsize=8)

plt.tight_layout()
fig_path = f'{FIGURES_DIR}/morphological_integrity_R2.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

# ── Summary table ─────────────────────────────────────────────────────────
df_summary = pd.DataFrame(summary_rows)
df_pivot   = df_summary.pivot(index='Variable',
                               columns='Source',
                               values='R²').round(4)

print('\n── R² Summary Table ──────────────────────────────────────────')
print(df_pivot.to_string())

# Model selection guidance
print('\n── Model Selection Guidance ──────────────────────────────────')
mean_r2 = df_summary.groupby('Source')['R²'].mean().round(4)
best    = mean_r2.idxmax()
print(f'Mean R² across all variables:')
for src, r2v in mean_r2.items():
    flag = ' ← RECOMMENDED FOR SCRIPT 4' if src == best else ''
    print(f'  {src:15s}: {r2v:.4f}{flag}')

# Save summary to CSV
csv_path = f'{ROOT}/validation/R2_summary.csv'
df_summary.to_csv(csv_path, index=False)
print(f'\nR² summary saved → {csv_path}')

In [ ]:
# ── SCRIPT 3 — MORPHOLOGICAL INTEGRITY ASSESSMENT ────────────────────────
# Reads manual cross-section measurements from Excel
# Three sheets: 'Manual Image Measurements', 'Attention Unet', 'Segformer'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import pearsonr, linregress
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import os

from google.colab import drive
drive.mount('/content/drive')

ROOT         = '/content/drive/MyDrive/BRAHMA_MORPHO'
FIGURES_DIR  = f'{ROOT}/figures/validation'
OUTPUT_EXCEL = f'{ROOT}/validation/Morphological_Integrity.xlsx'
EXCEL_PATH   = f'{ROOT}/validation/MIM.xlsx'   # ← adjust filename if needed

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_EXCEL), exist_ok=True)

# ── Column rename map: Excel headers → internal script names ─────────────
COL_RENAME = {
    'CS_id'                     : 'CS_id',
    'Total Channel Width(km)'   : 'channel_width_km',
    'Total Bar Width(km)'       : 'bar_width_km',
    'Vegetated Bar Width(km)'   : 'veg_bar_width_km',
    'Number of Channel Threads' : 'n_channel_threads',
    'Number of Bar Units'       : 'n_bar_units',
}

def load_sheet(path, sheet_name):
    """Load one sheet, rename columns, validate expected columns exist."""
    df = pd.read_excel(path, sheet_name=sheet_name)
    df.columns = df.columns.str.strip()          # strip accidental whitespace
    df = df.rename(columns=COL_RENAME)
    missing = [c for c in COL_RENAME.values() if c not in df.columns]
    if missing:
        raise ValueError(f"Sheet '{sheet_name}' is missing columns: {missing}\n"
                         f"Found: {list(df.columns)}")
    return df[list(COL_RENAME.values())]         # keep only the 6 expected cols

# ── Load all three sheets ─────────────────────────────────────────────────
MANUAL = load_sheet(EXCEL_PATH, 'Manual Image Measurements')
ATTN   = load_sheet(EXCEL_PATH, 'Attention Unet')
SEG    = load_sheet(EXCEL_PATH, 'Segformer')

print('✓ Sheets loaded successfully.\n')
print(f'  Manual          : {len(MANUAL)} rows')
print(f'  Attention U-Net : {len(ATTN)} rows')
print(f'  SegFormer       : {len(SEG)} rows')
print('\n── MANUAL preview ───────────────────────────────────────────────')
print(MANUAL.to_string(index=False))

In [ ]:
# ── R² REGRESSION ANALYSIS ────────────────────────────────────────────────
# Run this after Script 3 has loaded MANUAL, ATTN, SEG

VARS = [
    ('channel_width_km',  'Channel Width (km)'),
    ('bar_width_km',      'Total Bar Width (km)'),
    ('veg_bar_width_km',  'Vegetated Bar Width (km)'),
    ('n_channel_threads', 'N Channel Threads'),
    ('n_bar_units',       'N Bar Units'),
]

SOURCES = [
    ('Attention U-Net', ATTN, '#E84393', 's'),
    ('SegFormer',       SEG,  '#4393E8', '^'),
]

fig, axes = plt.subplots(
    len(VARS), len(SOURCES),
    figsize=(14, 5 * len(VARS))
)
fig.suptitle(
    'Morphological Integrity Assessment — R² Regression\n'
    'Manual NIR Measurements vs Model Predictions (7 Cross-Sections)',
    fontsize=13, fontweight='bold', y=1.01
)

summary = []

for vi, (var, var_label) in enumerate(VARS):
    for si, (src_name, df_src, color, marker) in enumerate(SOURCES):
        ax = axes[vi, si]

        merged = MANUAL[['CS_id', var]].merge(
            df_src[['CS_id', var]], on='CS_id',
            suffixes=('_manual', '_pred')
        ).dropna()

        if len(merged) < 3:
            ax.text(0.5, 0.5, 'Insufficient data\n(< 3 valid rows)',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=10, color='red')
            ax.set_title(f'{src_name}\n{var_label}', fontsize=9)
            continue

        x = merged[f'{var}_manual'].values.astype(float)
        y = merged[f'{var}_pred'].values.astype(float)

        r, p  = pearsonr(x, y)
        r2    = r ** 2
        rmse  = np.sqrt(np.mean((y - x) ** 2))
        slope, inter, _, _, _ = linregress(x, y)

        ax.scatter(x, y, color=color, marker=marker,
                   s=90, edgecolors='white', linewidth=0.8, zorder=3)

        for _, row in merged.iterrows():
            ax.annotate(
                row['CS_id'],
                (row[f'{var}_manual'], row[f'{var}_pred']),
                fontsize=7, alpha=0.7,
                xytext=(4, 4), textcoords='offset points'
            )

        lims = [min(x.min(), y.min()) * 0.85,
                max(x.max(), y.max()) * 1.15]
        ax.plot(lims, lims, 'k--', alpha=0.5,
                linewidth=1.2, label='1:1', zorder=2)

        x_line = np.linspace(lims[0], lims[1], 100)
        ax.plot(x_line, slope * x_line + inter,
                color=color, linewidth=2.0, alpha=0.85,
                label=f'Fit  R²={r2:.3f}', zorder=2)

        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel(f'Manual — {var_label}', fontsize=9)
        ax.set_ylabel(f'{src_name} — {var_label}', fontsize=9)
        ax.set_title(
            f'{src_name} | {var_label}\n'
            f'R²={r2:.3f}   RMSE={rmse:.3f}   n={len(merged)}',
            fontsize=9, fontweight='bold'
        )
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')
        ax.tick_params(labelsize=8)

        summary.append({
            'Variable': var_label,
            'Source'  : src_name,
            'R²'      : round(r2, 4),
            'RMSE'    : round(rmse, 4),
            'Bias'    : round(float(np.mean(y - x)), 4),
            'N'       : len(merged),
        })

plt.tight_layout()
fig_path = f'{FIGURES_DIR}/morphological_integrity_R2.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')

# ── Summary table ─────────────────────────────────────────────────────────
df_sum = pd.DataFrame(summary)
print('\n── R² Summary ──────────────────────────────────────────────────')
print(df_sum.pivot(index='Variable', columns='Source', values='R²')
            .round(4).to_string())

print('\n── Mean R² per model ────────────────────────────────────────────')
mean_r2 = df_sum.groupby('Source')['R²'].mean().round(4)
best    = mean_r2.idxmax()
for src, r2v in mean_r2.items():
    flag = '  ← SELECTED FOR SCRIPT 4' if src == best else ''
    print(f'  {src:16s}: {r2v:.4f}{flag}')

df_sum.to_csv(f'{ROOT}/validation/R2_summary.csv', index=False)
print(f'\nSummary saved → {ROOT}/validation/R2_summary.csv')

In [ ]:
# ── R² REGRESSION ANALYSIS — PUBLICATION FIGURES ─────────────────────────
# Two separate figures: one per model (Attention U-Net, SegFormer)
# Layout: 5 variables × 1 model per figure
# Style: Nature/AGU publication standard

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator, AutoMinorLocator
from scipy.stats import pearsonr, linregress
import os

# ── Publication style config ──────────────────────────────────────────────
plt.rcParams.update({
    'font.family'        : 'Arial',          # or 'Helvetica' if available
    'font.size'          : 8,
    'axes.labelsize'     : 8,
    'axes.titlesize'     : 8,
    'axes.linewidth'     : 0.6,
    'xtick.labelsize'    : 7,
    'ytick.labelsize'    : 7,
    'xtick.major.width'  : 0.6,
    'ytick.major.width'  : 0.6,
    'xtick.minor.width'  : 0.4,
    'ytick.minor.width'  : 0.4,
    'xtick.major.size'   : 3,
    'ytick.major.size'   : 3,
    'xtick.minor.size'   : 1.5,
    'ytick.minor.size'   : 1.5,
    'xtick.direction'    : 'out',
    'ytick.direction'    : 'out',
    'legend.fontsize'    : 7,
    'legend.frameon'     : False,
    'figure.dpi'         : 300,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'savefig.pad_inches' : 0.05,
    'lines.linewidth'    : 1.0,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
})

# ── Panel labels (a–e) ────────────────────────────────────────────────────
PANEL_LABELS = ['a', 'b', 'c', 'd', 'e']

# ── Variable definitions ──────────────────────────────────────────────────
VARS = [
    ('channel_width_km',  'Total channel width',    'km'),
    ('bar_width_km',      'Total bar width',         'km'),
    ('veg_bar_width_km',  'Vegetated bar width',     'km'),
    ('n_channel_threads', 'Channel thread count',    ''),
    ('n_bar_units',       'Bar unit count',          ''),
]

# ── Color palette (colorblind-safe) ──────────────────────────────────────
COLORS = {
    'Attention U-Net': {'scatter': '#C0392B', 'line': '#922B21', 'fill': '#FADBD8'},
    'SegFormer'      : {'scatter': '#1A5276', 'line': '#154360', 'fill': '#D6EAF8'},
}
MARKER = {
    'Attention U-Net': 'o',
    'SegFormer'      : 's',
}

SOURCES = [
    ('Attention U-Net', ATTN),
    ('SegFormer',       SEG),
]

summary = []

# ═════════════════════════════════════════════════════════════════════════
# One figure per model
# ═════════════════════════════════════════════════════════════════════════
for src_name, df_src in SOURCES:

    col   = COLORS[src_name]
    mkr   = MARKER[src_name]
    n_var = len(VARS)

    # ── Figure layout: 3 cols × 2 rows, last panel centred ────────────
    # Row 1: panels a b c  |  Row 2: panels d e (centred)
    fig = plt.figure(figsize=(6.8, 4.6))       # fits 2-column journal width

    gs_top = gridspec.GridSpec(
        1, 3,
        figure=fig,
        top=0.91, bottom=0.54,
        left=0.08, right=0.98,
        wspace=0.38,
    )
    gs_bot = gridspec.GridSpec(
        1, 4,                                   # 4-unit grid for centring 2
        figure=fig,
        top=0.44, bottom=0.07,
        left=0.08, right=0.98,
        wspace=0.38,
    )

    axes = [
        fig.add_subplot(gs_top[0, 0]),          # a – channel width
        fig.add_subplot(gs_top[0, 1]),          # b – bar width
        fig.add_subplot(gs_top[0, 2]),          # c – veg bar width
        fig.add_subplot(gs_bot[0, 0:2]),        # d – n threads (left half)
        fig.add_subplot(gs_bot[0, 2:4]),        # e – n bar units (right half)
    ]

    # ── Figure-level title ────────────────────────────────────────────
    fig.suptitle(
        f'Morphological integrity assessment — {src_name}',
        fontsize=9, fontweight='bold', x=0.5, y=0.98, ha='center',
    )

    for vi, (var, var_label, unit) in enumerate(VARS):
        ax = axes[vi]

        merged = MANUAL[['CS_id', var]].merge(
            df_src[['CS_id', var]], on='CS_id',
            suffixes=('_manual', '_pred')
        ).dropna()

        # ── Insufficient data guard ───────────────────────────────────
        if len(merged) < 3:
            ax.text(
                0.5, 0.5,
                'No data',
                ha='center', va='center',
                transform=ax.transAxes,
                fontsize=8, color='#999999',
                style='italic',
            )
            ax.set_title(
                f'({PANEL_LABELS[vi]})  {var_label}',
                loc='left', fontsize=8, fontweight='bold', pad=4,
            )
            continue

        x = merged[f'{var}_manual'].values.astype(float)
        y = merged[f'{var}_pred'].values.astype(float)

        r, p              = pearsonr(x, y)
        r2                = r ** 2
        rmse              = np.sqrt(np.mean((y - x) ** 2))
        bias              = float(np.mean(y - x))
        slope, inter, *_ = linregress(x, y)

        # ── Axis limits with symmetric padding ───────────────────────
        lo = min(x.min(), y.min())
        hi = max(x.max(), y.max())
        pad = (hi - lo) * 0.12 if (hi - lo) > 0 else 0.5
        lims = [lo - pad, hi + pad]

        # ── 1:1 line ─────────────────────────────────────────────────
        ax.plot(
            lims, lims,
            color='#666666', linewidth=0.8,
            linestyle='--', zorder=1,
            label='1:1 line',
        )

        # ── Confidence band on regression ────────────────────────────
        x_fit = np.linspace(lims[0], lims[1], 200)
        y_fit = slope * x_fit + inter
        n     = len(x)
        se    = np.sqrt(
            np.sum((y - (slope * x + inter)) ** 2) / (n - 2)
        ) * np.sqrt(
            1 / n + (x_fit - x.mean()) ** 2 / np.sum((x - x.mean()) ** 2)
        )
        ax.fill_between(
            x_fit, y_fit - 1.96 * se, y_fit + 1.96 * se,
            color=col['fill'], alpha=0.55, zorder=2,
        )

        # ── Regression line ───────────────────────────────────────────
        ax.plot(
            x_fit, y_fit,
            color=col['line'], linewidth=1.2,
            zorder=3, label='OLS fit',
        )

        # ── Scatter points ────────────────────────────────────────────
        ax.scatter(
            x, y,
            color=col['scatter'], marker=mkr,
            s=22, linewidths=0.4, edgecolors='white',
            zorder=4,
        )

        # ── Cross-section ID labels ───────────────────────────────────
        for _, row in merged.iterrows():
            ax.annotate(
                row['CS_id'].replace('CS_', ''),     # short: 01…07
                xy=(row[f'{var}_manual'], row[f'{var}_pred']),
                xytext=(3, 3), textcoords='offset points',
                fontsize=5.5, color='#444444', zorder=5,
            )

        # ── Stats annotation box ──────────────────────────────────────
        p_str = f'{p:.3f}' if p >= 0.001 else '<0.001'
        stats_txt = (
            f'$R^2$ = {r2:.3f}\n'
            f'RMSE = {rmse:.3f}{(" " + unit) if unit else ""}\n'
            f'Bias = {bias:+.3f}{(" " + unit) if unit else ""}\n'
            f'$p$ = {p_str}   $n$ = {n}'
        )
        ax.text(
            0.04, 0.97, stats_txt,
            transform=ax.transAxes,
            fontsize=6.2, va='top', ha='left',
            linespacing=1.55,
            color='#222222',
            bbox=dict(
                boxstyle='round,pad=0.3',
                facecolor='white',
                edgecolor='#cccccc',
                linewidth=0.5,
                alpha=0.85,
            ),
        )

        # ── Axes formatting ───────────────────────────────────────────
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_aspect('equal', adjustable='box')

        unit_str = f' ({unit})' if unit else ''
        ax.set_xlabel(f'Observed — {var_label}{unit_str}', labelpad=3)
        ax.set_ylabel(f'Predicted — {var_label}{unit_str}', labelpad=3)

        # Integer ticks for count variables
        if 'n_' in var:
            ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
            ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))
        else:
            ax.xaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5, prune='both'))

        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))

        # ── Panel label (a, b, c …) top-left ─────────────────────────
        ax.set_title(
            f'({PANEL_LABELS[vi]})  {var_label}',
            loc='left', fontsize=8, fontweight='bold', pad=4,
        )

        # ── Legend (first panel only to avoid clutter) ────────────────
        if vi == 0:
            handles, labels_leg = ax.get_legend_handles_labels()
            ax.legend(
                handles, labels_leg,
                loc='lower right',
                handlelength=1.5,
                handletextpad=0.4,
                borderpad=0.4,
            )

        summary.append({
            'Model'   : src_name,
            'Variable': var_label,
            'R²'      : round(r2, 4),
            'RMSE'    : round(rmse, 4),
            'Bias'    : round(bias, 4),
            'p-value' : round(p, 4),
            'N'       : n,
        })

    # ── Save ──────────────────────────────────────────────────────────
    safe_name = src_name.replace(' ', '_').replace('-', '').lower()
    fig_path  = f'{FIGURES_DIR}/morpho_integrity_{safe_name}.png'
    pdf_path  = f'{FIGURES_DIR}/morpho_integrity_{safe_name}.pdf'

    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path,           bbox_inches='tight')   # vector for journal
    plt.show()
    print(f'Saved → {fig_path}')
    print(f'Saved → {pdf_path}')

# ═════════════════════════════════════════════════════════════════════════
# Summary table + model selection
# ═════════════════════════════════════════════════════════════════════════
df_sum = pd.DataFrame(summary)

print('\n── R² per variable ──────────────────────────────────────────────')
print(
    df_sum.pivot(index='Variable', columns='Model', values='R²')
          .round(4).to_string()
)

print('\n── Mean R² per model ────────────────────────────────────────────')
mean_r2 = df_sum.groupby('Model')['R²'].mean().round(4)
best    = mean_r2.idxmax()
for mdl, r2v in mean_r2.items():
    flag = '  ← SELECTED FOR SCRIPT 4' if mdl == best else ''
    print(f'  {mdl:18s}: {r2v:.4f}{flag}')

df_sum.to_csv(f'{ROOT}/validation/R2_summary.csv', index=False)
print(f'\nSummary CSV → {ROOT}/validation/R2_summary.csv')